In [35]:
import pandas as pd
import numpy as np

file_path = "../data/raw/PS_20174392719_1491204439457_log.csv"
df = pd.read_csv(file_path)
print(df.shape)

X = df.drop(columns=["isFraud"])
y = df["isFraud"]

features_to_drop = [
    "nameOrig",
    "nameDest",
    "newbalanceOrig",
    "newbalanceDest",
    "isFlaggedFraud"
]

X = X.drop(columns=features_to_drop)

#one hot encode 
X = pd.get_dummies(X, columns=["type"], dtype=int)


#recreate our three engineered features
X["amount_to_origin_balance"] = (
    X["amount"] / (X["oldbalanceOrg"] + 1)
)
X["amount_to_destination_balance"] = (
    X["amount"] / (X["oldbalanceDest"] + 1)
)
X["log_amount"] = np.log1p(X["amount"])

#verify 
print(X.shape)
print(X.columns.tolist())


(6362620, 11)
(6362620, 12)
['step', 'amount', 'oldbalanceOrg', 'oldbalanceDest', 'type_CASH_IN', 'type_CASH_OUT', 'type_DEBIT', 'type_PAYMENT', 'type_TRANSFER', 'amount_to_origin_balance', 'amount_to_destination_balance', 'log_amount']


In [36]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (5090096, 12)
X_test : (1272524, 12)
y_train: (5090096,)
y_test : (1272524,)


In [37]:
print("Training class distribution:")
print(y_train.value_counts())

print("\nTesting class distribution:")
print(y_test.value_counts())

Training class distribution:
isFraud
0    5083526
1       6570
Name: count, dtype: int64

Testing class distribution:
isFraud
0    1270881
1       1643
Name: count, dtype: int64


In [38]:
from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)

In [39]:
logistic_model.fit(X_train, y_train)

,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to u

In [40]:
y_pred = logistic_model.predict(X_test)
y_pred[:10]

array([1, 0, 0, 0, 0, 1, 1, 0, 1, 0])

In [41]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print(cm)

[[1031840  239041]
 [     75    1568]]


In [42]:
from sklearn.metrics import precision_score, recall_score, f1_score

precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

Precision: 0.006516796961044682
Recall: 0.9543517954960439
F1 Score: 0.012945197562868419


In [43]:
from sklearn.metrics import roc_auc_score, average_precision_score

y_prob = logistic_model.predict_proba(X_test)[:, 1]

roc_auc = roc_auc_score(y_test, y_prob)
pr_auc = average_precision_score(y_test, y_prob)

print("ROC-AUC:", roc_auc)
print("PR-AUC:", pr_auc)

ROC-AUC: 0.9611621973244306
PR-AUC: 0.08373923308371672


In [44]:
thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

for threshold in thresholds:
    y_pred_threshold = (y_prob >= threshold).astype(int)

    precision = precision_score(y_test, y_pred_threshold)
    recall = recall_score(y_test, y_pred_threshold)
    f1 = f1_score(y_test, y_pred_threshold)

    print(
        f"Threshold: {threshold} | "
        f"Precision: {precision:.4f} | "
        f"Recall: {recall:.4f} | "
        f"F1: {f1:.4f}"
    )

Threshold: 0.1 | Precision: 0.0043 | Recall: 0.9988 | F1: 0.0085
Threshold: 0.2 | Precision: 0.0046 | Recall: 0.9988 | F1: 0.0092
Threshold: 0.3 | Precision: 0.0049 | Recall: 0.9982 | F1: 0.0097
Threshold: 0.4 | Precision: 0.0053 | Recall: 0.9848 | F1: 0.0106
Threshold: 0.5 | Precision: 0.0065 | Recall: 0.9544 | F1: 0.0129
Threshold: 0.6 | Precision: 0.0095 | Recall: 0.8977 | F1: 0.0187
Threshold: 0.7 | Precision: 0.0159 | Recall: 0.8016 | F1: 0.0312
Threshold: 0.8 | Precision: 0.0301 | Recall: 0.6506 | F1: 0.0576
Threshold: 0.9 | Precision: 0.0584 | Recall: 0.4699 | F1: 0.1038


In [45]:
'''The Logistic Regression model has good ranking ability:

ROC-AUC = 0.961

But when we actually convert its probabilities into fraud / normal predictions, the separation is not strong enough to produce high precision.

This is particularly visible here:

At threshold 0.9, only 5.84% of transactions predicted as fraud are actually fraud.
So simply changing the threshold isn't going to completely solve the problem.
Don't choose a final threshold yet

This is important for our project.

We're still at the Logistic Regression baseline. We have not compared it with the tree-based models yet.

Our next step should be:

Train Random Forest → evaluate it using the same metrics → compare with Logistic Regression.

That gives us a meaningful model comparison rather than trying to optimize the baseline prematurely.

One more thing

Your current Logistic Regression results should be recorded in Notebook 05:

Model: Logistic Regression
Class Weight: balanced

ROC-AUC: 0.9612
PR-AUC: 0.0837
Precision: 0.0065
Recall: 0.9544
F1: 0.0129

Don't modify the Logistic Regression model yet.'''

"The Logistic Regression model has good ranking ability:\n\nROC-AUC = 0.961\n\nBut when we actually convert its probabilities into fraud / normal predictions, the separation is not strong enough to produce high precision.\n\nThis is particularly visible here:\n\nAt threshold 0.9, only 5.84% of transactions predicted as fraud are actually fraud.\nSo simply changing the threshold isn't going to completely solve the problem.\nDon't choose a final threshold yet\n\nThis is important for our project.\n\nWe're still at the Logistic Regression baseline. We have not compared it with the tree-based models yet.\n\nOur next step should be:\n\nTrain Random Forest → evaluate it using the same metrics → compare with Logistic Regression.\n\nThat gives us a meaningful model comparison rather than trying to optimize the baseline prematurely.\n\nOne more thing\n\nYour current Logistic Regression results should be recorded in Notebook 05:\n\nModel: Logistic Regression\nClass Weight: balanced\n\nROC-AUC: 0

In [46]:
#we'll train the Random Forest model on the same X_train, X_test, y_train, and y_test.
from sklearn.ensemble import RandomForestClassifier
random_forest_model = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

In [47]:
random_forest_model.fit(X_train, y_train)

,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"class_weight class_weight: {""balanced"", ""balanced_subsample""}, dict or list of dicts, default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``The ""balanced_subsample"" mode is the same as ""balanced"" except thatweights are computed based on the bootstrap sample for every treegrown.For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.",'balanced'
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_fe

In [48]:
y_pred_rf = random_forest_model.predict(X_test)
y_pred_rf[:10]

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [49]:
from sklearn.metrics import confusion_matrix

cm_rf = confusion_matrix(y_test, y_pred_rf)

cm_rf

array([[1270879,       2],
       [      5,    1638]])

In [50]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

precision_rf = precision_score(y_test, y_pred_rf)
recall_rf = recall_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf)

print("Precision:", precision_rf)
print("Recall:", recall_rf)
print("F1 Score:", f1_rf)

Precision: 0.998780487804878
Recall: 0.996956786366403
F1 Score: 0.997867803837953


In [51]:
y_prob_rf = random_forest_model.predict_proba(X_test)[:, 1]
from sklearn.metrics import roc_auc_score, average_precision_score

roc_auc_rf = roc_auc_score(y_test, y_prob_rf)
pr_auc_rf = average_precision_score(y_test, y_prob_rf)

print("ROC-AUC:", roc_auc_rf)
print("PR-AUC:", pr_auc_rf)

ROC-AUC: 0.998780921492476
PR-AUC: 0.9975670877305878


In [52]:
feature_importance = pd.Series(
    random_forest_model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

feature_importance

amount_to_origin_balance         0.414572
oldbalanceOrg                    0.131438
amount_to_destination_balance    0.077258
amount                           0.070845
type_PAYMENT                     0.058606
log_amount                       0.056231
type_CASH_OUT                    0.053570
type_CASH_IN                     0.047228
type_TRANSFER                    0.045334
step                             0.025386
oldbalanceDest                   0.018832
type_DEBIT                       0.000701
dtype: float64

In [53]:
model_results = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "Precision": [precision, precision_rf],
    "Recall": [recall, recall_rf],
    "F1": [f1, f1_rf],
    "ROC-AUC": [roc_auc, roc_auc_rf],
    "PR-AUC": [pr_auc, pr_auc_rf]
})

model_results

,Model,Precision,Recall,F1,ROC-AUC,PR-AUC
0,Logistic Regression,0.058357,0.469872,0.103819,0.961162,0.083739
1,Random Forest,0.998780,0.996957,0.997868,0.998781,0.997567


In [54]:
import xgboost as xgb

print(xgb.__version__)

3.4.1


In [56]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

In [57]:
xgb_model.fit(X_train, y_train)

,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.8
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'logloss'
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [58]:
y_pred_xgb = xgb_model.predict(X_test)

In [59]:
y_pred_xgb[:10]

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [60]:
from sklearn.metrics import confusion_matrix

cm_xgb = confusion_matrix(y_test, y_pred_xgb)

cm_xgb

array([[1270757,     124],
       [    230,    1413]])

In [61]:
from sklearn.metrics import precision_score, recall_score, f1_score

precision_xgb = precision_score(y_test, y_pred_xgb)
recall_xgb = recall_score(y_test, y_pred_xgb)
f1_xgb = f1_score(y_test, y_pred_xgb)

print("Precision:", precision_xgb)
print("Recall:", recall_xgb)
print("F1 Score:", f1_xgb)

Precision: 0.9193233571893299
Recall: 0.8600121728545344
F1 Score: 0.8886792452830189


In [62]:
from sklearn.metrics import roc_auc_score, average_precision_score

y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]

roc_auc_xgb = roc_auc_score(y_test, y_prob_xgb)
pr_auc_xgb = average_precision_score(y_test, y_prob_xgb)

print("ROC-AUC:", roc_auc_xgb)
print("PR-AUC:", pr_auc_xgb)

ROC-AUC: 0.9956361615644276
PR-AUC: 0.9251984991379574


In [63]:
df["step"].min(), df["step"].max()

(np.int64(1), np.int64(743))

In [64]:
train_cutoff = int(df["step"].quantile(0.8))

train_cutoff

train_mask = df["step"] <= train_cutoff
test_mask = df["step"] > train_cutoff

X_train_time = X[train_mask]
X_test_time = X[test_mask]

y_train_time = y[train_mask]
y_test_time = y[test_mask]

print("X_train_time:", X_train_time.shape)
print("X_test_time:", X_test_time.shape)

print("\nTraining target:")
print(y_train_time.value_counts())

print("\nTesting target:")
print(y_test_time.value_counts())


X_train_time: (5113884, 12)
X_test_time: (1248736, 12)

Training target:
isFraud
0    5109921
1       3963
Name: count, dtype: int64

Testing target:
isFraud
0    1244486
1       4250
Name: count, dtype: int64


In [65]:
from xgboost import XGBClassifier

xgb_time_model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)
xgb_time_model.fit(X_train_time, y_train_time)

,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.8
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'logloss'
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [66]:
y_pred_xgb_time = xgb_time_model.predict(X_test_time)


In [67]:
from sklearn.metrics import confusion_matrix

cm_xgb_time = confusion_matrix(
    y_test_time,
    y_pred_xgb_time
)

cm_xgb_time

array([[1244420,      66],
       [   1123,    3127]])

In [68]:
from sklearn.metrics import precision_score, recall_score, f1_score

precision_xgb_time = precision_score(
    y_test_time,
    y_pred_xgb_time
)

recall_xgb_time = recall_score(
    y_test_time,
    y_pred_xgb_time
)

f1_xgb_time = f1_score(
    y_test_time,
    y_pred_xgb_time
)

print("Precision:", precision_xgb_time)
print("Recall:", recall_xgb_time)
print("F1 Score:", f1_xgb_time)

Precision: 0.9793297839022862
Recall: 0.735764705882353
F1 Score: 0.8402525863227194


In [69]:
from sklearn.metrics import roc_auc_score, average_precision_score

y_prob_xgb_time = xgb_time_model.predict_proba(
    X_test_time
)[:, 1]

roc_auc_xgb_time = roc_auc_score(
    y_test_time,
    y_prob_xgb_time
)

pr_auc_xgb_time = average_precision_score(
    y_test_time,
    y_prob_xgb_time
)

print("ROC-AUC:", roc_auc_xgb_time)
print("PR-AUC:", pr_auc_xgb_time)

ROC-AUC: 0.9869785490461406
PR-AUC: 0.9409506377300096


In [70]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = np.arange(0.1, 1.0, 0.1)

for threshold in thresholds:
    y_pred_threshold = (
        y_prob_xgb_time >= threshold
    ).astype(int)

    precision = precision_score(
        y_test_time,
        y_pred_threshold
    )

    recall = recall_score(
        y_test_time,
        y_pred_threshold
    )

    f1 = f1_score(
        y_test_time,
        y_pred_threshold
    )

    print(
        f"Threshold: {threshold:.1f} | "
        f"Precision: {precision:.4f} | "
        f"Recall: {recall:.4f} | "
        f"F1: {f1:.4f}"
    )

Threshold: 0.1 | Precision: 0.8903 | Recall: 0.8499 | F1: 0.8696
Threshold: 0.2 | Precision: 0.9548 | Recall: 0.7904 | F1: 0.8648
Threshold: 0.3 | Precision: 0.9716 | Recall: 0.7555 | F1: 0.8500
Threshold: 0.4 | Precision: 0.9759 | Recall: 0.7433 | F1: 0.8439
Threshold: 0.5 | Precision: 0.9793 | Recall: 0.7358 | F1: 0.8403
Threshold: 0.6 | Precision: 0.9802 | Recall: 0.7320 | F1: 0.8381
Threshold: 0.7 | Precision: 0.9807 | Recall: 0.7186 | F1: 0.8294
Threshold: 0.8 | Precision: 0.9845 | Recall: 0.6579 | F1: 0.7887
Threshold: 0.9 | Precision: 0.9890 | Recall: 0.5071 | F1: 0.6704


In [71]:
for threshold in thresholds:
    y_pred_threshold = (
        y_prob_xgb_time >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_test_time,
        y_pred_threshold
    ).ravel()

    print(
        f"Threshold: {threshold:.1f} | "
        f"TP: {tp} | FP: {fp} | "
        f"FN: {fn} | TN: {tn}"
    )

Threshold: 0.1 | TP: 3612 | FP: 445 | FN: 638 | TN: 1244041
Threshold: 0.2 | TP: 3359 | FP: 159 | FN: 891 | TN: 1244327
Threshold: 0.3 | TP: 3211 | FP: 94 | FN: 1039 | TN: 1244392
Threshold: 0.4 | TP: 3159 | FP: 78 | FN: 1091 | TN: 1244408
Threshold: 0.5 | TP: 3127 | FP: 66 | FN: 1123 | TN: 1244420
Threshold: 0.6 | TP: 3111 | FP: 63 | FN: 1139 | TN: 1244423
Threshold: 0.7 | TP: 3054 | FP: 60 | FN: 1196 | TN: 1244426
Threshold: 0.8 | TP: 2796 | FP: 44 | FN: 1454 | TN: 1244442
Threshold: 0.9 | TP: 2155 | FP: 24 | FN: 2095 | TN: 1244462


In [72]:
random_forest_time_model = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

random_forest_time_model.fit(
    X_train_time,
    y_train_time
)

,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"class_weight class_weight: {""balanced"", ""balanced_subsample""}, dict or list of dicts, default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``The ""balanced_subsample"" mode is the same as ""balanced"" except thatweights are computed based on the bootstrap sample for every treegrown.For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.",'balanced'
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_fe

In [73]:
#saving the model to to a file using joblib
import joblib 
joblib.dump(
    random_forest_time_model,
    "../models/random_forest_fraud_model.pkl"
)

['../models/random_forest_fraud_model.pkl']

In [85]:
feature_names = X.columns.tolist()

joblib.dump(
    feature_names,
    "../models/feature_names.pkl"
)

['../models/feature_names.pkl']

In [74]:
y_pred_rf_time = random_forest_time_model.predict(X_test_time)

In [75]:
from sklearn.metrics import confusion_matrix

cm_rf_time = confusion_matrix(y_test_time, y_pred_rf_time)

cm_rf_time

array([[1244484,       2],
       [      7,    4243]])

In [76]:
from sklearn.metrics import precision_score, recall_score, f1_score

precision_rf_time = precision_score(y_test_time, y_pred_rf_time)
recall_rf_time = recall_score(y_test_time, y_pred_rf_time)
f1_rf_time = f1_score(y_test_time, y_pred_rf_time)

print("Precision:", precision_rf_time)
print("Recall:", recall_rf_time)
print("F1 Score:", f1_rf_time)

Precision: 0.9995288574793875
Recall: 0.9983529411764706
F1 Score: 0.9989405532666275


In [77]:
from sklearn.metrics import roc_auc_score, average_precision_score

y_prob_rf_time = random_forest_time_model.predict_proba(X_test_time)[:, 1]

roc_auc_rf_time = roc_auc_score(y_test_time, y_prob_rf_time)
pr_auc_rf_time = average_precision_score(y_test_time, y_prob_rf_time)

print("ROC-AUC:", roc_auc_rf_time)
print("PR-AUC:", pr_auc_rf_time)

ROC-AUC: 0.9999998419380512
PR-AUC: 0.999945051253234


In [78]:
'''Now we can compare the two models fairly

Both were tested on the same future/time-based test set:

Metric	Random Forest	XGBoost
Precision	99.95%	97.93%
Recall	99.84%	73.58%
F1	99.89%	84.03%
ROC-AUC	99.99998%	98.70%
PR-AUC	99.9945%	94.10%

There is a very large difference here.

However, before we declare Random Forest our final model,
 I want to do one important check: make sure there isn't some dataset-specific leakage/artifact
 causing the exceptionally high Random Forest performance.'''

"Now we can compare the two models fairly\n\nBoth were tested on the same future/time-based test set:\n\nMetric\tRandom Forest\tXGBoost\nPrecision\t99.95%\t97.93%\nRecall\t99.84%\t73.58%\nF1\t99.89%\t84.03%\nROC-AUC\t99.99998%\t98.70%\nPR-AUC\t99.9945%\t94.10%\n\nThere is a very large difference here.\n\nHowever, before we declare Random Forest our final model,\n I want to do one important check: make sure there isn't some dataset-specific leakage/artifact\n causing the exceptionally high Random Forest performance."

In [79]:
#We should inspect Random Forest feature importance on the time-based model.
feature_importance_rf_time = pd.DataFrame({
    "Feature": X_train_time.columns,
    "Importance": random_forest_time_model.feature_importances_
})

feature_importance_rf_time = feature_importance_rf_time.sort_values(
    by="Importance",
    ascending=False
)

feature_importance_rf_time

,Feature,Importance
9,amount_to_origin_balance,0.435089
2,oldbalanceOrg,0.138934
10,amount_to_destination_balance,0.072411
1,amount,0.070523
5,type_CASH_OUT,0.060020
7,type_PAYMENT,0.056881
11,log_amount,0.054064
4,type_CASH_IN,0.045393
8,type_TRANSFER,0.043770
3,oldbalanceDest,0.016539


In [52]:
ratio_analysis = X.copy()
ratio_analysis["isFraud"] = y

ratio_analysis.groupby("isFraud")["amount_to_origin_balance"].describe()

,count,mean,std,min,25%,50%,75%,max
isFraud,,,,,,,,
0,6354407.0,70764.320934,508745.258992,1.765481e-08,0.233401,6.511566,12355.590000,92445516.64
1,8213.0,1161.966671,32297.153000,0.000000e+00,0.999991,0.999998,0.999999,1933920.80


In [80]:
'''This result is very important. It explains why amount_to_origin_balance is so powerful.

What we found

For normal transactions (isFraud = 0):

Median ratio ≈ 6.51
75th percentile ≈ 12,355.59
Very large maximum values exist.

For fraud (isFraud = 1):

Median ≈ 0.999998
75th percentile ≈ 0.999999

So almost all fraud transactions have:

amount ≈ oldbalanceOrg

That means the transaction is approximately equal to the sender's entire pre-transaction balance.

Why the Random Forest detects fraud so well

The model can learn a pattern roughly like:

amount / oldbalanceOrg ≈ 1
             ↓
       strong fraud signal

This explains why:
This result is very important. It explains why amount_to_origin_balance is so powerful.

What we found

For normal transactions (isFraud = 0):

Median ratio ≈ 6.51
75th percentile ≈ 12,355.59
Very large maximum values exist.

For fraud (isFraud = 1):

Median ≈ 0.999998
75th percentile ≈ 0.999999

So almost all fraud transactions have:

amount ≈ oldbalanceOrg

That means the transaction is approximately equal to the sender's entire pre-transaction balance.

Why the Random Forest detects fraud so well

The model can learn a pattern roughly like:

amount / oldbalanceOrg ≈ 1
             ↓
       strong fraud signal

This explains why:
amount_to_origin_balance
        ↓
43.5% feature importance
        ↓
Random Forest
        ↓
extremely high performance
But there's an important issue

Notice the normal maximum:

92,445,516.64

while fraud values are mostly around 1.

The ratio can become enormous when oldbalanceOrg is very small.

So this feature is very powerful, but we shouldn't automatically assume that the 99.99% performance means our model is universally excellent. It may be exploiting a very strong pattern specific to the PaySim simulation.

That's actually something you can discuss in an interview:

“The PaySim dataset contains a very strong relationship between fraud and the transaction amount relative to the sender's pre-transaction balance. The Random Forest exploited this feature effectively. However, because PaySim is synthetic, I would not assume the same performance would transfer directly to real-world transaction data.”

That's a good project limitation to mention.'''

"This result is very important. It explains why amount_to_origin_balance is so powerful.\n\nWhat we found\n\nFor normal transactions (isFraud = 0):\n\nMedian ratio ≈ 6.51\n75th percentile ≈ 12,355.59\nVery large maximum values exist.\n\nFor fraud (isFraud = 1):\n\nMedian ≈ 0.999998\n75th percentile ≈ 0.999999\n\nSo almost all fraud transactions have:\n\namount ≈ oldbalanceOrg\n\nThat means the transaction is approximately equal to the sender's entire pre-transaction balance.\n\nWhy the Random Forest detects fraud so well\n\nThe model can learn a pattern roughly like:\n\namount / oldbalanceOrg ≈ 1\n             ↓\n       strong fraud signal\n\nThis explains why:\nThis result is very important. It explains why amount_to_origin_balance is so powerful.\n\nWhat we found\n\nFor normal transactions (isFraud = 0):\n\nMedian ratio ≈ 6.51\n75th percentile ≈ 12,355.59\nVery large maximum values exist.\n\nFor fraud (isFraud = 1):\n\nMedian ≈ 0.999998\n75th percentile ≈ 0.999999\n\nSo almost all fr

In [53]:
fraud_ratio = X.loc[y == 1, "amount_to_origin_balance"]

print("Fraud transactions:", len(fraud_ratio))
print("Ratio >= 0.99:", (fraud_ratio >= 0.99).sum())
print("Ratio >= 0.999:", (fraud_ratio >= 0.999).sum())
print("Ratio >= 0.9999:", (fraud_ratio >= 0.9999).sum())

Fraud transactions: 8213
Ratio >= 0.99: 8045
Ratio >= 0.999: 8005
Ratio >= 0.9999: 7785


In [ ]:
'''That confirms the pattern very clearly.

Out of 8,213 fraud transactions:

8,045 (97.95%) have ratio ≥ 0.99
8,005 (97.47%) have ratio ≥ 0.999
7,785 (94.79%) have ratio ≥ 0.9999

So the feature is extremely discriminative in this PaySim dataset.

What this means for your project

This explains a large part of the Random Forest's exceptional performance:

amount_to_origin_balance
          ↓
~98% of fraud cases ≈ 1 or higher
          ↓
Random Forest learns this pattern
          ↓
Very high precision / recall

This doesn't mean your model is wrong. But it does mean you should be careful about saying:

“My model achieves 99.99% performance in real-world fraud detection.”

❌ Don't make that claim.

Instead:

“On the PaySim time-based holdout set, Random Forest achieved 99.95% precision, 99.84% recall, 99.89% F1, 99.99998% ROC-AUC and 99.9945% PR-AUC. The strong performance is partly driven by a highly discriminative transaction-to-origin-balance feature present in the synthetic dataset.”

That's a much stronger interview answer, because you're showing that you understand the result rather than simply presenting impressive numbers.

Where we are now

We've finished the important investigation. ✅

Next we can move to:

Model comparison → threshold analysis → final model → interpretation → saving model → Streamlit app.

So the next actual coding step is creating the final comparison table for Random Forest vs XGBoost on the time-based test set.'''

In [81]:
#Final time based model comparision
model_comparison_time = pd.DataFrame({
    "Model": ["Random Forest", "XGBoost"],
    "Precision": [precision_rf_time, 0.9793297839],
    "Recall": [recall_rf_time, 0.7357647059],
    "F1": [f1_rf_time, 0.8402525863],
    "ROC-AUC": [roc_auc_rf_time, 0.9869785490],
    "PR-AUC": [pr_auc_rf_time, 0.9409506377]
})

model_comparison_time

,Model,Precision,Recall,F1,ROC-AUC,PR-AUC
0,Random Forest,0.999529,0.998353,0.998941,1.000000,0.999945
1,XGBoost,0.979330,0.735765,0.840253,0.986979,0.940951


In [82]:
#final time-based comparison
'''| Metric    | Random Forest | XGBoost |
| --------- | ------------: | ------: |
| Precision |    **99.95%** |  97.93% |
| Recall    |    **99.84%** |  73.58% |
| F1        |    **99.89%** |  84.03% |
| ROC-AUC   |     **~100%** |  98.70% |
| PR-AUC    |    **99.99%** |  94.10% |
'''


'| Metric    | Random Forest | XGBoost |\n| --------- | ------------: | ------: |\n| Precision |    **99.95%** |  97.93% |\n| Recall    |    **99.84%** |  73.58% |\n| F1        |    **99.89%** |  84.03% |\n| ROC-AUC   |     **~100%** |  98.70% |\n| PR-AUC    |    **99.99%** |  94.10% |\n'

In [83]:
'''Random Forest threshold analysis
So far, your Random Forest metrics use the default 0.5 threshold.
We need to see what happens when we change the threshold, just like we did with XGBoost.'''

thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

for threshold in thresholds:
    y_pred_threshold = (y_prob_rf_time >= threshold).astype(int)

    precision = precision_score(y_test_time, y_pred_threshold)
    recall = recall_score(y_test_time, y_pred_threshold)
    f1 = f1_score(y_test_time, y_pred_threshold)

    print(
        f"Threshold: {threshold} | "
        f"Precision: {precision:.4f} | "
        f"Recall: {recall:.4f} | "
        f"F1: {f1:.4f}"
    )

Threshold: 0.1 | Precision: 0.9902 | Recall: 0.9998 | F1: 0.9950
Threshold: 0.2 | Precision: 0.9958 | Recall: 0.9998 | F1: 0.9978
Threshold: 0.3 | Precision: 0.9991 | Recall: 0.9998 | F1: 0.9994
Threshold: 0.4 | Precision: 0.9993 | Recall: 0.9998 | F1: 0.9995
Threshold: 0.5 | Precision: 0.9995 | Recall: 0.9991 | F1: 0.9993
Threshold: 0.6 | Precision: 0.9998 | Recall: 0.9880 | F1: 0.9938
Threshold: 0.7 | Precision: 1.0000 | Recall: 0.9616 | F1: 0.9804
Threshold: 0.8 | Precision: 1.0000 | Recall: 0.8645 | F1: 0.9273
Threshold: 0.9 | Precision: 1.0000 | Recall: 0.4715 | F1: 0.6409


In [84]:
'''What this tells us

The important pattern is:

0.1–0.4: extremely high recall, while precision also remains above 99%.
0.4: precision 99.93%, recall 99.98%, F1 99.95%.
0.5: precision increases slightly to 99.95%, but recall drops slightly to 99.91%.
0.7+: precision reaches 100%, but recall starts falling significantly.
0.9: you miss more than half of the fraud cases.

So the threshold is controlling the trade-off between:

Lower threshold → catch more fraud, but potentially flag more normal transactions.
Higher threshold → fewer false alarms, but potentially miss more fraud.

For our project

I would not permanently hard-code 0.4 yet.

Instead, in the next stage we should document something like:

Operating threshold: 0.4 (candidate)
On the time-based holdout set, this threshold produced 99.93% precision,
 99.98% recall and 99.95% F1.

Then make the threshold configurable in the Streamlit application.

This is better for a real fraud-detection project because the acceptable balance 
between false positives and missed fraud depends on the business requirement.

One important observation

Our time-based Random Forest baseline at threshold 0.5 was already:

Precision: 99.95%
Recall: 99.84%
F1: 99.89%
ROC-AUC: ~100%
PR-AUC: 99.9945%

The threshold analysis shows that 0.3–0.5 all perform exceptionally well on this PaySim holdout.

The next step should therefore be Notebook 06: clean model comparison, 
using the same time-based test set and consistent evaluation procedure for Random Forest and XGBoost.'''

'What this tells us\n\nThe important pattern is:\n\n0.1–0.4: extremely high recall, while precision also remains above 99%.\n0.4: precision 99.93%, recall 99.98%, F1 99.95%.\n0.5: precision increases slightly to 99.95%, but recall drops slightly to 99.91%.\n0.7+: precision reaches 100%, but recall starts falling significantly.\n0.9: you miss more than half of the fraud cases.\n\nSo the threshold is controlling the trade-off between:\n\nLower threshold → catch more fraud, but potentially flag more normal transactions.\nHigher threshold → fewer false alarms, but potentially miss more fraud.\n\nFor our project\n\nI would not permanently hard-code 0.4 yet.\n\nInstead, in the next stage we should document something like:\n\nOperating threshold: 0.4 (candidate)\nOn the time-based holdout set, this threshold produced 99.93% precision,\n 99.98% recall and 99.95% F1.\n\nThen make the threshold configurable in the Streamlit application.\n\nThis is better for a real fraud-detection project becaus